# Estructura del notebook
1. Realizar benchmark de generacion de trazas no procesadas (timestamp, hex data separadas por espacios)
2. Separacion de datos por codigo de funcion

In [2]:
import pandas as pd
import glob
import os
from datetime import datetime

# Buscar todos los archivos de trazas procesadas
ruta_busqueda = "./data/output/processed_traces/pr_ts-*.csv"
archivos = sorted(glob.glob(ruta_busqueda), key=os.path.getmtime, reverse=True)

if not archivos:
    raise FileNotFoundError(f"No se encontraron archivos en {ruta_busqueda}")

# Seleccionar el archivo más reciente (el primero tras ordenar inversamente por tiempo)
archivo_reciente = archivos[0]
print(f"[+] Cargando el archivo más reciente: {archivo_reciente}")

# Cargar los datos en memoria
df_principal = pd.read_csv(archivo_reciente)
print(f"[+] Total de trazas cargadas: {len(df_principal)}")

[+] Cargando el archivo más reciente: ./data/output/processed_traces\pr_ts-20260913_014748.csv
[+] Total de trazas cargadas: 18525


In [3]:
import numpy as np

# 1. Crear el directorio de separación con la fecha actual
fecha_actual = datetime.now().strftime("%Y%m%d_%H%M%S")
directorio_salida = f"./data/separations/{fecha_actual}"
os.makedirs(directorio_salida, exist_ok=True)
print(f"[+] Directorio creado: {directorio_salida}\n")

# Obtener todos los códigos de función únicos en el dataset
codigos_presentes = df_principal['function_code'].dropna().unique()

for fc in codigos_presentes:
    # 2. Aislar las trazas de este código de función específico
    df_fc = df_principal[df_principal['function_code'] == fc].copy()
    
    # 3. Eliminar columnas que contengan exclusivamente valores -1
    # Se evalúa columna por columna: si TODOS sus datos son -1, se descarta para este FC
    columnas_validas = [col for col in df_fc.columns if not (df_fc[col] == -1).all()]
    df_fc_limpio = df_fc[columnas_validas]
    
    # 4. Generar y guardar el archivo CSV
    archivo_out = f"{directorio_salida}/dataset_fc_{int(fc)}.csv"
    df_fc_limpio.to_csv(archivo_out, index=False)
    
    # Imprimir el resumen de la limpieza
    columnas_eliminadas = len(df_fc.columns) - len(df_fc_limpio.columns)
    print(f" -> FC {int(fc):02d}: {len(df_fc_limpio)} trazas retenidas.")
    print(f"          Columnas activas: {len(df_fc_limpio.columns)} (Se eliminaron {columnas_eliminadas} columnas vacías)")

print(f"\n[+] Separación completada. Todos los archivos están en: {directorio_salida}")

[+] Directorio creado: ./data/separations/20260913_133731

 -> FC 04: 2958 trazas retenidas.
          Columnas activas: 8 (Se eliminaron 24 columnas vacías)
 -> FC 16: 1482 trazas retenidas.
          Columnas activas: 30 (Se eliminaron 2 columnas vacías)
 -> FC 03: 5899 trazas retenidas.
          Columnas activas: 8 (Se eliminaron 24 columnas vacías)
 -> FC 01: 2942 trazas retenidas.
          Columnas activas: 8 (Se eliminaron 24 columnas vacías)
 -> FC 02: 2954 trazas retenidas.
          Columnas activas: 8 (Se eliminaron 24 columnas vacías)
 -> FC 05: 872 trazas retenidas.
          Columnas activas: 8 (Se eliminaron 24 columnas vacías)
 -> FC 06: 1418 trazas retenidas.
          Columnas activas: 8 (Se eliminaron 24 columnas vacías)

[+] Separación completada. Todos los archivos están en: ./data/separations/20260913_133731


## Entrenamiento por codigo de funcion
Pasos: 
Separacion de codigo de funcion
Implementacio de correlacion por codigo de funcion
Implementacion de Isolation Forest

### Separacion y analisis de correlacion por codigo de funcion 1-6

In [ ]:
import pandas as pd
import glob
import os
import numpy as np

def analizar_autocorrelacion_lectura(fc_objetivo, ruta_busqueda="./data/output/processed_traces/pr_ts-*.csv"):
    """
    Escanea el archivo procesado más reciente y calcula la autocorrelación 
    específicamente para los códigos de función del 1 al 6.
    """
    if not (1 <= fc_objetivo <= 6):
        print(f"[-] Advertencia: El código {fc_objetivo} está fuera del rango 1-6 para esta función.")
        
    # 1. Buscar y cargar el archivo procesado más reciente
    archivos = sorted(glob.glob(ruta_busqueda), key=os.path.getmtime, reverse=True)

    if not archivos:
        raise FileNotFoundError("No se encontraron archivos procesados.")

    df_principal = pd.read_csv(archivos[0])

    # 2. Filtrar únicamente el Código de Función especificado
    df_fc = df_principal[df_principal['function_code'] == fc_objetivo].copy()

    if df_fc.empty:
        print(f"[-] No hay trazas con Function Code {fc_objetivo} en el archivo.")
        return
        
    print(f"[+] Analizando {len(df_fc)} trazas del Function Code {fc_objetivo}...")
    
    # Asegurar el orden cronológico
    df_fc['timestamp'] = pd.to_datetime(df_fc['timestamp'])
    df_fc = df_fc.sort_values('timestamp')
    
    # Columnas exclusivas para análisis de lectura/escritura simple
    columnas_analisis = ['address_unit_1', 'quantity_unit_1', 'delta_t']
    
    def encontrar_mejor_patron(serie, max_lag=30):
        if serie.std() == 0 or len(serie.dropna()) < max_lag:
            return None, 0.0
            
        mejor_lag = 0
        mejor_score = -1.0
        
        for lag in range(1, max_lag + 1):
            score = serie.autocorr(lag=lag)
            if not np.isnan(score) and score > mejor_score:
                mejor_score = score
                mejor_lag = lag
                
        return mejor_lag, mejor_score

    print(f"\n=== Resultados de Autocorrelación para FC {fc_objetivo} ===")
    
    for col in columnas_analisis:
        if col in df_fc.columns:
            # Reemplazar posibles -1 con NaN para que no distorsionen la matemática
            serie_limpia = df_fc[col].replace(-1, np.nan).dropna()
            
            n_estimado, score = encontrar_mejor_patron(serie_limpia)
            
            print(f"Variable: {col}")
            if n_estimado:
                print(f"  └─ Ventana N detectada : {n_estimado}")
                print(f"  └─ Score de confianza  : {score:.4f} (-1 a 1)")
                
                # Evaluación de utilidad del patrón
                if score > 0.8:
                    print("  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)")
                elif score > 0.4:
                    print("  └─ Utilidad            : MEDIA (Patrón ruidoso o con variaciones)")
                else:
                    print("  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)")
            else:
                print("  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).")
            print("-" * 40)
    

In [2]:
analizar_autocorrelacion_lectura(1)

[+] Analizando 2942 trazas del Function Code 1...

=== Resultados de Autocorrelación para FC 1 ===
Variable: address_unit_1
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9973 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------
Variable: quantity_unit_1
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9973 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------
Variable: delta_t
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9431 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------


In [3]:
analizar_autocorrelacion_lectura(2)

[+] Analizando 2954 trazas del Function Code 2...

=== Resultados de Autocorrelación para FC 2 ===
Variable: address_unit_1
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9959 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------
Variable: quantity_unit_1
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9959 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------
Variable: delta_t
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.8797 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------


In [18]:
analizar_autocorrelacion_lectura(3)

[+] Analizando 5899 trazas del Function Code 3...

=== Resultados de Autocorrelación para FC 3 ===
Variable: address_unit_1
  └─ Ventana N detectada : 4
  └─ Score de confianza  : 0.9923 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------
Variable: quantity_unit_1
  └─ Ventana N detectada : 4
  └─ Score de confianza  : 0.9230 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------
Variable: delta_t
  └─ Ventana N detectada : 4
  └─ Score de confianza  : 0.9506 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------


In [19]:
analizar_autocorrelacion_lectura(4)

[+] Analizando 2958 trazas del Function Code 4...

=== Resultados de Autocorrelación para FC 4 ===
Variable: address_unit_1
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9973 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------
Variable: quantity_unit_1
  └─ Ventana N detectada : 16
  └─ Score de confianza  : 0.6409 (-1 a 1)
  └─ Utilidad            : MEDIA (Patrón ruidoso o con variaciones)
----------------------------------------
Variable: delta_t
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9565 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------


In [20]:
analizar_autocorrelacion_lectura(5)

[+] Analizando 872 trazas del Function Code 5...

=== Resultados de Autocorrelación para FC 5 ===
Variable: address_unit_1
  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).
----------------------------------------
Variable: quantity_unit_1
  └─ Ventana N detectada : 1
  └─ Score de confianza  : 0.5012 (-1 a 1)
  └─ Utilidad            : MEDIA (Patrón ruidoso o con variaciones)
----------------------------------------
Variable: delta_t
  └─ Ventana N detectada : 2
  └─ Score de confianza  : 0.9174 (-1 a 1)
  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)
----------------------------------------


In [21]:
analizar_autocorrelacion_lectura(6)

[+] Analizando 1418 trazas del Function Code 6...

=== Resultados de Autocorrelación para FC 6 ===
Variable: address_unit_1
  └─ Ventana N detectada : 1
  └─ Score de confianza  : 0.3282 (-1 a 1)
  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)
----------------------------------------
Variable: quantity_unit_1
  └─ Ventana N detectada : 1
  └─ Score de confianza  : 0.4515 (-1 a 1)
  └─ Utilidad            : MEDIA (Patrón ruidoso o con variaciones)
----------------------------------------
Variable: delta_t
  └─ Ventana N detectada : 18
  └─ Score de confianza  : 0.5965 (-1 a 1)
  └─ Utilidad            : MEDIA (Patrón ruidoso o con variaciones)
----------------------------------------


### Separacion y analisis de correlacion por codigo de funcion 15, 16 y 23

In [5]:
import pandas as pd
import glob
import os
import numpy as np

# 1. Buscar y cargar el archivo procesado más reciente
ruta_busqueda = "./data/output/processed_traces/pr_ts-*.csv"
archivos = sorted(glob.glob(ruta_busqueda), key=os.path.getmtime, reverse=True)

if not archivos:
    raise FileNotFoundError("No se encontraron archivos procesados.")

df_principal = pd.read_csv(archivos[0])

# 2. Filtrar únicamente el Código de Función 16
df_fc16 = df_principal[df_principal['function_code'] == 16].copy()

if df_fc16.empty:
    print("[-] No hay trazas con Function Code 16 en el archivo.")
else:
    print(f"[+] Analizando {len(df_fc16)} trazas del Function Code 16...")
    
    # Asegurar el orden cronológico
    df_fc16['timestamp'] = pd.to_datetime(df_fc16['timestamp'])
    df_fc16 = df_fc16.sort_values('timestamp')
    
    # Seleccionar todas las columnas excepto las metadatas especificadas
    cols_excluir = ['timestamp', 'slave_address', 'function_code', 'function_name', 'crc_calculated']
    columnas_analisis = [col for col in df_fc16.columns if col not in cols_excluir]
    
    def encontrar_mejor_patron(serie, max_lag=30):
        """Calcula la autocorrelación hasta max_lag y devuelve el pico más alto."""
        if serie.std() == 0 or len(serie.dropna()) < max_lag:
            return None, 0.0
            
        mejor_lag = 0
        mejor_score = -1.0
        
        for lag in range(1, max_lag + 1):
            score = serie.autocorr(lag=lag)
            if not np.isnan(score) and score > mejor_score:
                mejor_score = score
                mejor_lag = lag
                
        return mejor_lag, mejor_score

    print("\n=== Resultados de Autocorrelación para FC 16 ===")
    
    for col in columnas_analisis:
        # Reemplazar posibles -1 con NaN
        serie_limpia = df_fc16[col].replace(-1, np.nan).dropna()
        
        n_estimado, score = encontrar_mejor_patron(serie_limpia)
        
        print(f"Variable: {col}")
        if n_estimado:
            print(f"  └─ Ventana N detectada : {n_estimado}")
            print(f"  └─ Score de confianza  : {score:.4f} (-1 a 1)")
            
            # Evaluación de utilidad del patrón
            if score > 0.8:
                print("  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)")
            elif score > 0.4:
                print("  └─ Utilidad            : MEDIA (Patrón ruidoso o con variaciones)")
            else:
                print("  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)")
        else:
            print("  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).")
        print("-" * 40)

[+] Analizando 1482 trazas del Function Code 16...

=== Resultados de Autocorrelación para FC 16 ===
Variable: address_unit_1
  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).
----------------------------------------
Variable: quantity_unit_1
  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).
----------------------------------------
Variable: count_unit_1
  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).
----------------------------------------
Variable: mininum_value_register
  └─ Ventana N detectada : 29
  └─ Score de confianza  : 0.0596 (-1 a 1)
  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)
----------------------------------------
Variable: maximum_value_register
  └─ Ventana N detectada : 9
  └─ Score de confianza  : 0.0483 (-1 a 1)
  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)
----------------------------------------
Variable: tmean_value_register
  └─ Ventana N d

## Funciones de correlacion

Para codigo de funcion 15, 16 y 23

In [11]:
import pandas as pd
import glob
import os
import numpy as np

def analizar_autocorrelacion_fc(fc_objetivo, ruta_busqueda="./data/output/processed_traces/pr_ts-*.csv"):
    """
    Escanea el archivo procesado más reciente, aísla un código de función
    y calcula el mejor 'lag' de autocorrelación para todas sus variables útiles.
    """
    # 1. Buscar y cargar el archivo procesado más reciente
    archivos = sorted(glob.glob(ruta_busqueda), key=os.path.getmtime, reverse=True)

    if not archivos:
        raise FileNotFoundError("No se encontraron archivos procesados.")

    df_principal = pd.read_csv(archivos[0])

    # 2. Filtrar únicamente el Código de Función especificado
    df_fc = df_principal[df_principal['function_code'] == fc_objetivo].copy()

    if df_fc.empty:
        print(f"[-] No hay trazas con Function Code {fc_objetivo} en el archivo.")
        return
        
    print(f"[+] Analizando {len(df_fc)} trazas del Function Code {fc_objetivo}...")
    
    # Asegurar el orden cronológico
    df_fc['timestamp'] = pd.to_datetime(df_fc['timestamp'])
    df_fc = df_fc.sort_values('timestamp')
    
    # Seleccionar todas las columnas excepto las metadatas especificadas
    cols_excluir = ['timestamp', 'slave_address', 'function_code', 'function_name', 'crc_calculated']
    columnas_analisis = [col for col in df_fc.columns if col not in cols_excluir]
    
    def encontrar_mejor_patron(serie, max_lag=30):
        if serie.std() == 0 or len(serie.dropna()) < max_lag:
            return None, 0.0
            
        mejor_lag = 0
        mejor_score = -1.0
        
        for lag in range(1, max_lag + 1):
            score = serie.autocorr(lag=lag)
            if not np.isnan(score) and score > mejor_score:
                mejor_score = score
                mejor_lag = lag
                
        return mejor_lag, mejor_score

    print(f"\n=== Resultados de Autocorrelación para FC {fc_objetivo} ===")
    
    for col in columnas_analisis:
        # Reemplazar posibles -1 con NaN
        serie_limpia = df_fc[col].replace(-1, np.nan).dropna()
        
        n_estimado, score = encontrar_mejor_patron(serie_limpia)
        
        print(f"Variable: {col}")
        if n_estimado:
            print(f"  └─ Ventana N detectada : {n_estimado}")
            print(f"  └─ Score de confianza  : {score:.4f} (-1 a 1)")
            
            # Evaluación de utilidad del patrón
            if score > 0.8:
                print("  └─ Utilidad            : ALTA (Patrón muy repetitivo y cíclico)")
            elif score > 0.4:
                print("  └─ Utilidad            : MEDIA (Patrón ruidoso o con variaciones)")
            else:
                print("  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)")
        else:
            print("  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).")
        print("-" * 40)

In [12]:
analizar_autocorrelacion_fc(15)

[-] No hay trazas con Function Code 15 en el archivo.


In [13]:
analizar_autocorrelacion_fc(16)

[+] Analizando 1482 trazas del Function Code 16...

=== Resultados de Autocorrelación para FC 16 ===
Variable: address_unit_1
  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).
----------------------------------------
Variable: quantity_unit_1
  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).
----------------------------------------
Variable: count_unit_1
  └─ No se pudo detectar un patrón (Varianza cero o datos insuficientes).
----------------------------------------
Variable: mininum_value_register
  └─ Ventana N detectada : 29
  └─ Score de confianza  : 0.0596 (-1 a 1)
  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)
----------------------------------------
Variable: maximum_value_register
  └─ Ventana N detectada : 9
  └─ Score de confianza  : 0.0483 (-1 a 1)
  └─ Utilidad            : BAJA (Datos caóticos o sin ciclo definido)
----------------------------------------
Variable: tmean_value_register
  └─ Ventana N d

In [14]:
analizar_autocorrelacion_fc(23)

[-] No hay trazas con Function Code 23 en el archivo.


## Entrenamiento mediante correlacion

In [32]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

# 1. Configuración de rutas e hiperparámetros
# Reemplaza la fecha con la carpeta exacta donde está tu archivo
directorio_entrada = "./data/separations/20260913_133731"
archivo_entrada = f"{directorio_entrada}/dataset_fc_1.csv"
N_VENTANA = 2

print(f"[+] Cargando datos desde: {archivo_entrada}")
df_raw = pd.read_csv(archivo_entrada)
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])

# 2. Seleccionar exclusivamente las columnas de utilidad
columnas_base = ['timestamp', 'address_unit_1', 'quantity_unit_1', 'delta_t']
df_util = df_raw[columnas_base].copy()

print(f"[+] Aplicando ventanas deslizantes (N={N_VENTANA})...")
rolling_win = df_util[['address_unit_1', 'quantity_unit_1', 'delta_t']].rolling(window=N_VENTANA, min_periods=N_VENTANA)

# 3. Construcción del DataFrame Secuencial
df_secuencial = pd.DataFrame(index=df_util.index)
# Conservamos el timestamp de la ÚLTIMA traza que cierra la ventana
df_secuencial['timestamp_cierre_ventana'] = df_util['timestamp']

# --- Características de Tiempo ---
df_secuencial['delta_t_std'] = rolling_win['delta_t'].std()
df_secuencial['delta_t_mean'] = rolling_win['delta_t'].mean()
df_secuencial['delta_t_sum'] = rolling_win['delta_t'].sum() # Tiempo total del ciclo

# --- Características de Direcciones (Memoria) ---
df_secuencial['address_rango'] = rolling_win['address_unit_1'].max() - rolling_win['address_unit_1'].min()
# Conteo de direcciones únicas leídas en este ciclo (Optimizado para C con numpy)
df_secuencial['address_unicas'] = rolling_win['address_unit_1'].apply(lambda x: len(np.unique(x)), raw=True)

# --- Características de Cantidad (Payload) ---
df_secuencial['quantity_total'] = rolling_win['quantity_unit_1'].sum()
df_secuencial['quantity_std'] = rolling_win['quantity_unit_1'].std().fillna(0.0)

# 4. Limpieza (Eliminar las primeras N-1 filas que no pudieron formar una ventana completa)
df_secuencial = df_secuencial.dropna().reset_index(drop=True)

# 5. Exportación
fecha_actual = datetime.now().strftime("%Y%m%d_%H%M%S")
directorio_salida = f"./data/sequence/{fecha_actual}"
os.makedirs(directorio_salida, exist_ok=True)

archivo_salida = f"{directorio_salida}/sequence_fc_3_N{N_VENTANA}.csv"
df_secuencial.to_csv(archivo_salida, index=False)

print(f"\n[+] Proceso completado exitosamente.")
print(f"  └─ Trazas originales   : {len(df_raw)}")
print(f"  └─ Ventanas generadas  : {len(df_secuencial)}")
print(f"  └─ Archivo guardado en : {archivo_salida}")

# Vista previa de los datos de entrenamiento
print("\nVista previa de las primeras 3 ventanas de entrenamiento:")
print(df_secuencial.head(3))

[+] Cargando datos desde: ./data/separations/20260913_133731/dataset_fc_1.csv
[+] Aplicando ventanas deslizantes (N=2)...

[+] Proceso completado exitosamente.
  └─ Trazas originales   : 2942
  └─ Ventanas generadas  : 2941
  └─ Archivo guardado en : ./data/sequence/20260913_170239/sequence_fc_3_N2.csv

Vista previa de las primeras 3 ventanas de entrenamiento:
  timestamp_cierre_ventana  delta_t_std  delta_t_mean  delta_t_sum  \
0  2026-09-13 05:28:15.592     0.265165        0.2105        0.421   
1  2026-09-13 05:28:19.069     0.222739        0.1805        0.361   
2  2026-09-13 05:28:19.093     0.222032        0.1810        0.362   

   address_rango  address_unicas  quantity_total  quantity_std  
0          231.0             2.0         20875.0  14756.611417  
1          231.0             2.0         20875.0  14756.611417  
2          231.0             2.0         20875.0  14756.611417  


In [33]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import MinMaxScaler

def entrenar_modelos_secuenciales(ruta_csv_secuencia):
    """
    Carga el dataset de ventanas móviles, escala los datos (requerido para SVM)
    y entrena Isolation Forest y One-Class SVM.
    """
    print(f"[+] Cargando dataset de entrenamiento: {ruta_csv_secuencia}")
    df_train = pd.read_csv(ruta_csv_secuencia)
    
    # Separar características de metadatos (timestamp)
    X_train = df_train.drop(columns=['timestamp_cierre_ventana'], errors='ignore')
    columnas_modelo = X_train.columns.tolist()
    
    # Escalado estricto para One-Class SVM
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    # 1. Entrenar Isolation Forest (Usa datos crudos, no le afecta la escala)
    modelo_if = IsolationForest(contamination='auto', random_state=42, n_jobs=-1)
    modelo_if.fit(X_train)
    
    # 2. Entrenar One-Class SVM (Usa datos escalados)
    modelo_ocsvm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)
    modelo_ocsvm.fit(X_train_scaled)
    
    print("[+] Modelos entrenados exitosamente.")
    return modelo_if, modelo_ocsvm, scaler, columnas_modelo, df_train, X_train, X_train_scaled

# --- EJECUCIÓN ---
# Reemplaza 'fecha_actual' con el nombre de tu carpeta real
ruta_entrenamiento = "./data/sequence/20260913_165919/sequence_fc_3_N2.csv" 
modelo_if, modelo_ocsvm, scaler, columnas_entrenamiento, df_train, X_train, X_train_scaled = entrenar_modelos_secuenciales(ruta_entrenamiento)

[+] Cargando dataset de entrenamiento: ./data/sequence/20260913_165919/sequence_fc_3_N2.csv
[+] Modelos entrenados exitosamente.


In [35]:
def evaluar_buffer_tiempo_real(ruta_trazas_crudas, indice_inicio, N, modelo_if, modelo_ocsvm, scaler, columnas_entrenamiento):
    """
    Simula la intercepción de N trazas contiguas, replica la ingeniería de características
    y evalúa el vector resultante contra los modelos entrenados.
    """
    # 1. Simular la llegada de trazas crudas leyendo el archivo base
    df_raw = pd.read_csv(ruta_trazas_crudas)
    
    # Extraer estrictamente N trazas a partir del índice indicado
    df_buffer = df_raw.iloc[indice_inicio : indice_inicio + N].copy()
    
    if len(df_buffer) < N:
        print("[-] Error: No hay suficientes trazas en el índice especificado para llenar el buffer.")
        return
        
    print(f"\n=== INTERCEPTANDO {N} TRAZAS CONTIGUAS (Índices {indice_inicio} al {indice_inicio + N - 1}) ===")
    print(df_buffer[['timestamp', 'address_unit_1', 'quantity_unit_1', 'delta_t']])
    
    # 2. Ingeniería de Características (Debe ser matemáticamente idéntica al entrenamiento)
    vector_caracteristicas = {
        'delta_t_std': df_buffer['delta_t'].std(),
        'delta_t_mean': df_buffer['delta_t'].mean(),
        'delta_t_sum': df_buffer['delta_t'].sum(),
        'address_rango': df_buffer['address_unit_1'].max() - df_buffer['address_unit_1'].min(),
        'address_media': df_buffer['address_unit_1'].mean(),
        'address_unicas': len(df_buffer['address_unit_1'].unique()),
        'quantity_total': df_buffer['quantity_unit_1'].sum(),
        'quantity_std': df_buffer['quantity_unit_1'].std()
    }
    
    # Rellenar NaN por si la desviación estándar fallara con valores idénticos
    X_nuevo = pd.DataFrame([vector_caracteristicas]).fillna(0.0)
    
    # Alinear columnas para evitar errores de orden
    X_nuevo = X_nuevo[columnas_entrenamiento]
    
    print("\n[+] Vector estadístico extraído del buffer:")
    print(X_nuevo.iloc[0].to_string())
    
    # 3. Inferencia (Predicción)
    # Isolation Forest recibe datos crudos
    pred_if = modelo_if.predict(X_nuevo)[0]
    
    # One-Class SVM requiere que escalemos el vector nuevo usando los pesos originales
    X_nuevo_scaled = scaler.transform(X_nuevo)
    pred_ocsvm = modelo_ocsvm.predict(X_nuevo_scaled)[0]
    
    print("\n=== RESULTADOS DE DETECCIÓN ===")
    print(f"Isolation Forest : {'[ANOMALÍA]' if pred_if == -1 else 'Normal'}")
    print(f"One-Class SVM    : {'[ANOMALÍA]' if pred_ocsvm == -1 else 'Normal'}")

# --- EJECUCIÓN ---
ruta_cruda = "./data/separations/20260913_133731/dataset_fc_1.csv"
indice_prueba = 23# Cambia este número para extraer otro bloque de 4 trazas

# Descomentar para probar cuando los modelos estén entrenados:
evaluar_buffer_tiempo_real(ruta_cruda, indice_prueba, 2, modelo_if, modelo_ocsvm, scaler, columnas_entrenamiento)


=== INTERCEPTANDO 2 TRAZAS CONTIGUAS (Índices 23 al 24) ===
                  timestamp  address_unit_1  quantity_unit_1  delta_t
23  2026-09-13 05:28:54.104             256            20872    0.023
24  2026-09-13 05:28:57.581              25                3    0.398

[+] Vector estadístico extraído del buffer:
delta_t_std           0.265165
delta_t_mean          0.210500
delta_t_sum           0.421000
address_rango       231.000000
address_unicas        2.000000
quantity_total    20875.000000
quantity_std      14756.611417

=== RESULTADOS DE DETECCIÓN ===
Isolation Forest : Normal
One-Class SVM    : Normal


In [36]:
import pandas as pd
import numpy as np
from IPython.display import display

# ==========================================
# CONFIGURACIÓN DEL ENTORNO DE PRUEBA
# ==========================================
ruta_trazas = "./data/separations/20260913_133731/dataset_fc_1.csv"
indice_inicio = 5  # Modifica este índice para deslizar la ventana en el archivo
N_VENTANA = 2

# 1. Cargar datos originales y aislar N trazas contiguas
df_raw = pd.read_csv(ruta_trazas)
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])

if indice_inicio + N_VENTANA > len(df_raw):
    print(f"[-] Error: Índice fuera de rango. El archivo solo tiene {len(df_raw)} trazas.")
else:
    df_buffer = df_raw.iloc[indice_inicio : indice_inicio + N_VENTANA].copy()

    # 2. Recrear el vector de características exacto para estas 4 trazas
    vector_caracteristicas = {
        'delta_t_std': df_buffer['delta_t'].std(),
        'delta_t_mean': df_buffer['delta_t'].mean(),
        'delta_t_sum': df_buffer['delta_t'].sum(),
        'address_rango': df_buffer['address_unit_1'].max() - df_buffer['address_unit_1'].min(),
        'address_media': df_buffer['address_unit_1'].mean(),
        'address_unicas': len(df_buffer['address_unit_1'].unique()),
        'quantity_total': df_buffer['quantity_unit_1'].sum(),
        'quantity_std': df_buffer['quantity_unit_1'].std()
    }

    # Transformar a DataFrame y rellenar NaNs matemáticos
    X_nuevo = pd.DataFrame([vector_caracteristicas]).fillna(0.0)
    
    # Asegurar el mismo orden de columnas que se usó en el entrenamiento (asume que 'columnas_modelo' está en memoria)
    X_nuevo = X_nuevo[columnas_entrenamiento]

    # 3. Predicciones
    # Isolation Forest
    pred_if = modelo_if.predict(X_nuevo)[0]
    estado_if = "[ANOMALÍA]" if pred_if == -1 else "Normal"
    
    # One-Class SVM (Requiere el escalador entrenado previamente)
    X_nuevo_scaled = scaler.transform(X_nuevo)
    pred_ocsvm = modelo_ocsvm.predict(X_nuevo_scaled)[0]
    estado_ocsvm = "[ANOMALÍA]" if pred_ocsvm == -1 else "Normal"

    # 4. Inyectar resultados al DataFrame original para visualización conjunta
    df_buffer['Estado_IsolationForest'] = estado_if
    df_buffer['Estado_OneClassSVM'] = estado_ocsvm

    print(f"=== RESULTADOS DE INFERENCIA (Trazas del índice {indice_inicio} al {indice_inicio + N_VENTANA - 1}) ===")
    display(df_buffer[['timestamp', 'address_unit_1', 'quantity_unit_1', 'delta_t', 'Estado_IsolationForest', 'Estado_OneClassSVM']])

=== RESULTADOS DE INFERENCIA (Trazas del índice 5 al 6) ===


,timestamp,address_unit_1,quantity_unit_1,delta_t,Estado_IsolationForest,Estado_OneClassSVM
5,2026-09-13 05:28:22.593,256,20872,0.024,Normal,Normal
6,2026-09-13 05:28:26.069,25,3,0.398,Normal,Normal


In [37]:
import pandas as pd
import numpy as np

def analizar_resultados_entrenamiento(df_train, X_train, X_train_scaled, modelo_if, modelo_ocsvm):
    """
    Evalúa el dataset secuencial completo, extrae los porcentajes de anomalías
    y calcula el 'Score de Anomalía' para medir la severidad de las desviaciones.
    """
    df_resultados = df_train.copy()
    
    # 1. Obtener predicciones y Grado de Anomalía (Score)
    # Isolation Forest
    df_resultados['IF_Pred'] = modelo_if.predict(X_train)
    df_resultados['IF_Score'] = modelo_if.decision_function(X_train)
    df_resultados['IF_Estado'] = np.where(df_resultados['IF_Pred'] == 1, 'Normal', 'Anomalía')
    
    # One-Class SVM
    df_resultados['OCSVM_Pred'] = modelo_ocsvm.predict(X_train_scaled)
    df_resultados['OCSVM_Score'] = modelo_ocsvm.decision_function(X_train_scaled)
    df_resultados['OCSVM_Estado'] = np.where(df_resultados['OCSVM_Pred'] == 1, 'Normal', 'Anomalía')
    
    total_trazas = len(df_resultados)
    
    # 2. Resumen Estadístico - Isolation Forest
    if_normales = (df_resultados['IF_Pred'] == 1).sum()
    if_anomalas = (df_resultados['IF_Pred'] == -1).sum()
    if_pct_anomalas = (if_anomalas / total_trazas) * 100
    
    # 3. Resumen Estadístico - One-Class SVM
    ocsvm_normales = (df_resultados['OCSVM_Pred'] == 1).sum()
    ocsvm_anomalas = (df_resultados['OCSVM_Pred'] == -1).sum()
    ocsvm_pct_anomalas = (ocsvm_anomalas / total_trazas) * 100

    print("=== RESULTADOS GLOBALES DE ENTRENAMIENTO ===")
    print(f"Total de Secuencias Evaluadas : {total_trazas}\n")
    
    print("[ Isolation Forest ]")
    print(f"  └─ Normales  : {if_normales} ({100 - if_pct_anomalas:.2f}%)")
    print(f"  └─ Anomalías : {if_anomalas} ({if_pct_anomalas:.2f}%)")
    
    print("\n[ One-Class SVM ]")
    print(f"  └─ Normales  : {ocsvm_normales} ({100 - ocsvm_pct_anomalas:.2f}%)")
    print(f"  └─ Anomalías : {ocsvm_anomalas} ({ocsvm_pct_anomalas:.2f}%)\n")
    
    # 4. Aislar y mostrar las anomalías más severas
    anomalias_severas = df_resultados[df_resultados['IF_Pred'] == -1].sort_values(by='IF_Score').head(5)
    
    if not anomalias_severas.empty:
        print("=== TOP 5 ANOMALÍAS MÁS SEVERAS (Basado en IF_Score) ===")
        # Mostramos las métricas clave y el score de severidad
        columnas_ver = ['timestamp_cierre_ventana', 'delta_t_std', 'quantity_total', 'IF_Score', 'OCSVM_Score']
        print(anomalias_severas[columnas_ver].to_string(index=False))
        
    return df_resultados

# --- EJECUCIÓN ---
# Asumiendo que df_train, X_train, X_train_scaled y los modelos ya están cargados en la celda anterior
df_resultados_completos = analizar_resultados_entrenamiento(df_train, X_train, X_train_scaled, modelo_if, modelo_ocsvm)

=== RESULTADOS GLOBALES DE ENTRENAMIENTO ===
Total de Secuencias Evaluadas : 2941

[ Isolation Forest ]
  └─ Normales  : 2775 (94.36%)
  └─ Anomalías : 166 (5.64%)

[ One-Class SVM ]
  └─ Normales  : 2819 (95.85%)
  └─ Anomalías : 122 (4.15%)

=== TOP 5 ANOMALÍAS MÁS SEVERAS (Basado en IF_Score) ===
timestamp_cierre_ventana  delta_t_std  quantity_total  IF_Score  OCSVM_Score
 2026-09-13 06:45:23.066     0.689429             6.0 -0.353016  -126.108590
 2026-09-13 06:44:11.408     0.690136             6.0 -0.353016  -126.111967
 2026-09-13 06:45:16.931     0.000000             6.0 -0.341631  -125.736411
 2026-09-13 06:45:09.346     0.000707             6.0 -0.341062  -125.733511
 2026-09-13 06:44:54.176     0.000707             6.0 -0.341062  -125.733511
